# 116 — Herramientas tipadas y efectos laterales

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

Una **herramienta tipada** es un contrato de tres partes: descripción semántica (cuándo
usarla y cuándo NO), **JSON Schema** de entrada (tipos, required, enums, rangos — se
valida ANTES de ejecutar) y contrato de salida/errores (el error estructurado es parte
de la interfaz: el agente lo observa y decide con él).

**Taxonomía de efectos:** pura/solo lectura → escritura reversible → escritura
irreversible → efecto externo distribuido. La clase de efecto determina qué controles
exige (reintento libre, registro, aprobación humana, auditoría).

### 🔁 Idempotencia y dry-run

**Idempotente:** `f(f(x)) = f(x)` — repetir la operación no multiplica el efecto
(`set_price(10)` sí; `add_units(+5)` no). Importa porque los agentes REINTENTAN y un
timeout no dice si el efecto se aplicó. Técnica estándar: **clave de idempotencia** —
el servidor registra las claves aplicadas y convierte duplicados en no-ops.

**Dry-run:** con `dry_run: true` la herramienta valida precondiciones y devuelve QUÉ
haría (diff/plan/costo) sin aplicar nada. Convierte un efecto irreversible en dos
pasos: uno observable y uno autorizado.

El laboratorio `agent` usa dos herramientas puras (`status`, `sum`): reintentables sin
riesgo — el caso base contra el que se mide todo lo demás.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** `status` y `sum` no modifican nada: puras (y por tanto idempotentes).
Clasificación: `get_balance` pura/idempotente; `set_status(id,"closed")` escritura
reversible (se puede reabrir) e idempotente (fijar dos veces = una); `append_log`
escritura reversible pero NO idempotente (dos appends = dos líneas); `send_email`
irreversible y externa, NO idempotente; `delete_file` sin papelera: irreversible,
idempotente en estado final (el segundo borrado es no-op/404) — irreversibilidad e
idempotencia son ejes independientes; `upsert_config` reversible e idempotente.

**Ejercicio 2.** Ver celda de código. El campo extra es `idempotency_key` (string,
required cuando `dry_run=false`): sin él, un reintento tras timeout puede reembolsar
dos veces — el peor efecto posible en una herramienta de dinero.

**Ejercicio 3.** (a) `{"would_apply": false, "error": {"code": "INSUFFICIENT_STOCK",
"available": 3, "requested": 10}}` — nada cambió; (b) `{"would_apply": true,
"resulting": {"central": 0, "norte": 3}}` — ensayo, nada cambió; (c) `{"applied": true,
"resulting": {"central": 0, "norte": 3}}` — efecto aplicado y clave registrada;
(d) `{"applied": false, "duplicate_of": "k1", "resulting": {"central": 0, "norte": 3}}`
— no-op con el resultado original: el estado NO cambia otra vez.

**Ejercicio 4.** Ver celda: el registro guarda `key → resultado`; el duplicado devuelve
el resultado almacenado sin ejecutar la operación. El contador termina en 1 con dos
llamadas — exactamente la garantía que un agente necesita para reintentar a ciegas.

In [ ]:
result = run_lab("agent", seed=116)
assert result["kind"] == "agent"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 2 — schema de refund_order con reintento seguro
schema_refund = {
    "type": "object",
    "properties": {
        "order_id": {"type": "string", "pattern": "^ORD-[0-9]{6}$"},
        "amount": {"type": "number", "minimum": 0.01, "maximum": 10000},
        "reason": {"type": "string",
                   "enum": ["defective", "not_delivered", "customer_request"]},
        "dry_run": {"type": "boolean", "default": True},
        "idempotency_key": {"type": "string", "minLength": 8},
    },
    "required": ["order_id", "amount", "reason"],
}
# regla operativa: si dry_run == false, idempotency_key es obligatoria
print("campos:", list(schema_refund["properties"]))


In [ ]:
# Ejercicio 4 — registro de idempotencia demostrado
registro = {}

def apply(key, operation):
    if key in registro:
        return {"applied": False, "duplicate_of": key, "result": registro[key]}
    result = operation()
    registro[key] = result
    return {"applied": True, "result": result}

contador = {"n": 0}
def incrementar():
    contador["n"] += 1
    return contador["n"]

r1 = apply("k1", incrementar)
r2 = apply("k1", incrementar)  # reintento tras "timeout"
print(r1)
print(r2)
assert contador["n"] == 1          # el efecto NO se duplico
assert r2["result"] == r1["result"]  # el duplicado devuelve el resultado original


## Reflexión

1. Las dos herramientas del laboratorio son puras. ¿Qué tres mecanismos nuevos (schema,
   dry-run, idempotency_key) se vuelven obligatorios en cuanto una herramienta escribe
   estado, y qué riesgo concreto cubre cada uno?
2. ¿Por qué un timeout es el caso que separa "reintentar" de "reintentar con clave de
   idempotencia"? ¿Qué información NO te da un timeout?
3. Propón el error estructurado que debería devolver una herramienta `book_meeting`
   cuando la sala está ocupada, de modo que el siguiente thought pueda replantear sin
   intervención humana.